# Steel Surface Defect Detection - Inference Demo

This notebook demonstrates how to use trained YOLO and DETR models for steel defect detection.

**Contents:**
1. Setup and imports
2. Load trained models
3. Run inference on sample images
4. Visualize results
5. Compute per-image metrics

In [ ]:
# Install required packages (if not already installed)
# !pip install torch torchvision opencv-python matplotlib numpy ultralytics

In [ ]:
import sys
import os
from pathlib import Path

# Add src directory to path
sys.path.append(str(Path.cwd().parent / 'src'))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import torch
from ultralytics import YOLO

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Configuration

In [ ]:
# Paths
PROJECT_ROOT = Path.cwd().parent
YOLO_MODEL_PATH = PROJECT_ROOT / 'results' / 'models' / 'yolo_best.pt'
DETR_MODEL_PATH = PROJECT_ROOT / 'results' / 'models' / 'detr_best.pth'
TEST_IMAGES_DIR = PROJECT_ROOT / 'data' / 'processed' / 'yolo' / 'test' / 'images'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'visuals' / 'notebook_outputs'

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Class names
CLASS_NAMES = ['Crazing', 'Inclusion', 'Patches', 'Pitted_surface', 'Rolled-in_scale', 'Scratches']
CLASS_COLORS = [
    (255, 0, 0),      # Red
    (0, 255, 0),      # Green
    (0, 0, 255),      # Blue
    (255, 255, 0),    # Yellow
    (255, 0, 255),    # Magenta
    (0, 255, 255),    # Cyan
]

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

## 2. Load Models

In [ ]:
# Load YOLO model
if YOLO_MODEL_PATH.exists():
    yolo_model = YOLO(str(YOLO_MODEL_PATH))
    print(f"✓ YOLO model loaded from {YOLO_MODEL_PATH}")
else:
    print(f"✗ YOLO model not found at {YOLO_MODEL_PATH}")
    print("  Please train the model first using: python src/train_yolo.py")
    yolo_model = None

In [ ]:
# Load DETR model
if DETR_MODEL_PATH.exists():
    from train_detr import DETRLite
    
    detr_model = DETRLite(num_classes=6)
    checkpoint = torch.load(DETR_MODEL_PATH, map_location=DEVICE)
    detr_model.load_state_dict(checkpoint['model_state_dict'])
    detr_model = detr_model.to(DEVICE)
    detr_model.eval()
    print(f"✓ DETR model loaded from {DETR_MODEL_PATH}")
else:
    print(f"✗ DETR model not found at {DETR_MODEL_PATH}")
    print("  Please train the model first using: python src/train_detr.py")
    detr_model = None

## 3. Helper Functions

In [ ]:
def visualize_detections(image, boxes, scores, classes, class_names, title=""):
    """
    Visualize detections on image using matplotlib.
    
    Args:
        image: Input image (RGB)
        boxes: Bounding boxes [N, 4] in xyxy format
        scores: Confidence scores [N]
        classes: Class indices [N]
        class_names: List of class names
        title: Plot title
    """
    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(image)
    
    for box, score, cls in zip(boxes, scores, classes):
        x1, y1, x2, y2 = box
        w, h = x2 - x1, y2 - y1
        
        # Draw box
        color = np.array(CLASS_COLORS[int(cls) % len(CLASS_COLORS)]) / 255.0
        rect = Rectangle((x1, y1), w, h, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        
        # Draw label
        label = f"{class_names[int(cls)]}: {score:.2f}"
        ax.text(x1, y1 - 5, label, color='white', fontsize=10,
               bbox=dict(facecolor=color, alpha=0.8, edgecolor='none', pad=2))
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    return fig

In [ ]:
def run_yolo_inference(model, image_path, conf_thres=0.25):
    """
    Run YOLO inference on single image.
    
    Returns:
        boxes, scores, classes, image_rgb
    """
    results = model(image_path, conf=conf_thres, verbose=False)
    result = results[0]
    
    boxes = result.boxes.xyxy.cpu().numpy()
    scores = result.boxes.conf.cpu().numpy()
    classes = result.boxes.cls.cpu().numpy()
    
    # Load image for visualization
    image = cv2.imread(str(image_path))
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    return boxes, scores, classes, image_rgb

In [ ]:
def run_detr_inference(model, image_path, conf_thres=0.5, device='cuda'):
    """
    Run DETR inference on single image.
    
    Returns:
        boxes, scores, classes, image_rgb
    """
    import torchvision.transforms as T
    
    # Load image
    image = cv2.imread(str(image_path))
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    
    # Preprocess
    transform = T.Compose([T.ToTensor(), T.Resize((640, 640))])
    img_tensor = transform(image_rgb).unsqueeze(0).to(device)
    
    # Inference
    with torch.no_grad():
        outputs = model(img_tensor)
    
    # Post-process
    logits = outputs['pred_logits'][0]
    boxes_norm = outputs['pred_boxes'][0]
    
    probs = logits.softmax(-1)
    scores, labels = probs[:, :-1].max(-1)
    
    keep = scores > conf_thres
    scores = scores[keep].cpu().numpy()
    labels = labels[keep].cpu().numpy()
    boxes_norm = boxes_norm[keep].cpu().numpy()
    
    # Convert to pixel coordinates
    boxes = np.zeros_like(boxes_norm)
    boxes[:, 0] = (boxes_norm[:, 0] - boxes_norm[:, 2] / 2) * w
    boxes[:, 1] = (boxes_norm[:, 1] - boxes_norm[:, 3] / 2) * h
    boxes[:, 2] = (boxes_norm[:, 0] + boxes_norm[:, 2] / 2) * w
    boxes[:, 3] = (boxes_norm[:, 1] + boxes_norm[:, 3] / 2) * h
    
    return boxes, scores, labels, image_rgb

## 4. Load Test Images

In [ ]:
# Find test images
if TEST_IMAGES_DIR.exists():
    test_images = list(TEST_IMAGES_DIR.glob('*.jpg')) + list(TEST_IMAGES_DIR.glob('*.png'))
    test_images = test_images[:5]  # Select first 5 images
    print(f"Found {len(test_images)} test images")
    for img in test_images:
        print(f"  - {img.name}")
else:
    print(f"Test images directory not found: {TEST_IMAGES_DIR}")
    print("Please prepare the dataset first using: python src/data_utils.py --action prepare")
    test_images = []

## 5. Run Inference with YOLO

In [ ]:
if yolo_model and test_images:
    print("Running YOLO inference...\n")
    
    for i, img_path in enumerate(test_images, 1):
        print(f"Image {i}: {img_path.name}")
        
        # Run inference
        boxes, scores, classes, image_rgb = run_yolo_inference(yolo_model, img_path, conf_thres=0.25)
        
        print(f"  Detected {len(boxes)} defects")
        for j, (box, score, cls) in enumerate(zip(boxes, scores, classes)):
            print(f"    {j+1}. {CLASS_NAMES[int(cls)]}: {score:.3f}")
        
        # Visualize
        fig = visualize_detections(
            image_rgb, boxes, scores, classes, CLASS_NAMES,
            title=f"YOLOv8 Detection - {img_path.name}"
        )
        
        # Save
        output_file = OUTPUT_DIR / f"yolo_detection_{i}.png"
        fig.savefig(output_file, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"  Saved to: {output_file}\n")
else:
    print("Skipping YOLO inference (model or images not available)")

## 6. Run Inference with DETR

In [ ]:
if detr_model and test_images:
    print("Running DETR inference...\n")
    
    for i, img_path in enumerate(test_images, 1):
        print(f"Image {i}: {img_path.name}")
        
        # Run inference
        boxes, scores, classes, image_rgb = run_detr_inference(detr_model, img_path, conf_thres=0.5, device=DEVICE)
        
        print(f"  Detected {len(boxes)} defects")
        for j, (box, score, cls) in enumerate(zip(boxes, scores, classes)):
            print(f"    {j+1}. {CLASS_NAMES[int(cls)]}: {score:.3f}")
        
        # Visualize
        fig = visualize_detections(
            image_rgb, boxes, scores, classes, CLASS_NAMES,
            title=f"DETR Detection - {img_path.name}"
        )
        
        # Save
        output_file = OUTPUT_DIR / f"detr_detection_{i}.png"
        fig.savefig(output_file, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"  Saved to: {output_file}\n")
else:
    print("Skipping DETR inference (model or images not available)")

## 7. Side-by-Side Comparison

In [ ]:
if yolo_model and detr_model and test_images:
    print("Creating side-by-side comparisons...\n")
    
    for i, img_path in enumerate(test_images[:3], 1):  # Compare first 3 images
        # YOLO
        yolo_boxes, yolo_scores, yolo_classes, image_rgb = run_yolo_inference(yolo_model, img_path)
        
        # DETR
        detr_boxes, detr_scores, detr_classes, _ = run_detr_inference(detr_model, img_path, device=DEVICE)
        
        # Plot side by side
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
        
        # YOLO
        ax1.imshow(image_rgb)
        for box, score, cls in zip(yolo_boxes, yolo_scores, yolo_classes):
            x1, y1, x2, y2 = box
            w, h = x2 - x1, y2 - y1
            color = np.array(CLASS_COLORS[int(cls) % len(CLASS_COLORS)]) / 255.0
            rect = Rectangle((x1, y1), w, h, linewidth=2, edgecolor=color, facecolor='none')
            ax1.add_patch(rect)
            label = f"{CLASS_NAMES[int(cls)]}: {score:.2f}"
            ax1.text(x1, y1 - 5, label, color='white', fontsize=9,
                   bbox=dict(facecolor=color, alpha=0.8, edgecolor='none', pad=2))
        ax1.set_title(f"YOLOv8 ({len(yolo_boxes)} detections)", fontsize=14, fontweight='bold')
        ax1.axis('off')
        
        # DETR
        ax2.imshow(image_rgb)
        for box, score, cls in zip(detr_boxes, detr_scores, detr_classes):
            x1, y1, x2, y2 = box
            w, h = x2 - x1, y2 - y1
            color = np.array(CLASS_COLORS[int(cls) % len(CLASS_COLORS)]) / 255.0
            rect = Rectangle((x1, y1), w, h, linewidth=2, edgecolor=color, facecolor='none')
            ax2.add_patch(rect)
            label = f"{CLASS_NAMES[int(cls)]}: {score:.2f}"
            ax2.text(x1, y1 - 5, label, color='white', fontsize=9,
                   bbox=dict(facecolor=color, alpha=0.8, edgecolor='none', pad=2))
        ax2.set_title(f"DETR ({len(detr_boxes)} detections)", fontsize=14, fontweight='bold')
        ax2.axis('off')
        
        plt.suptitle(f"Comparison - {img_path.name}", fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        # Save
        output_file = OUTPUT_DIR / f"comparison_{i}.png"
        fig.savefig(output_file, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved comparison to: {output_file}")
else:
    print("Skipping comparison (both models required)")

## 8. Summary Statistics

In [ ]:
if yolo_model and detr_model and test_images:
    print("Computing summary statistics...\n")
    
    yolo_det_counts = []
    detr_det_counts = []
    
    for img_path in test_images:
        # YOLO
        yolo_boxes, _, _, _ = run_yolo_inference(yolo_model, img_path)
        yolo_det_counts.append(len(yolo_boxes))
        
        # DETR
        detr_boxes, _, _, _ = run_detr_inference(detr_model, img_path, device=DEVICE)
        detr_det_counts.append(len(detr_boxes))
    
    print(f"Total images processed: {len(test_images)}")
    print(f"\nYOLO Statistics:")
    print(f"  Total detections: {sum(yolo_det_counts)}")
    print(f"  Avg detections per image: {np.mean(yolo_det_counts):.2f}")
    print(f"  Min/Max: {min(yolo_det_counts)} / {max(yolo_det_counts)}")
    
    print(f"\nDETR Statistics:")
    print(f"  Total detections: {sum(detr_det_counts)}")
    print(f"  Avg detections per image: {np.mean(detr_det_counts):.2f}")
    print(f"  Min/Max: {min(detr_det_counts)} / {max(detr_det_counts)}")
else:
    print("Cannot compute statistics (models or images not available)")

## 9. Conclusion

This notebook demonstrated:
- Loading trained YOLO and DETR models
- Running inference on test images
- Visualizing detection results
- Comparing both architectures side-by-side

**Key Observations:**
- YOLO typically provides faster inference
- Both models can detect various defect types
- Performance varies by defect size and complexity

For production deployment, consider:
- Confidence threshold tuning based on your application
- Model optimization (ONNX, quantization) for edge devices
- Post-processing to filter/merge overlapping detections